# House Prices Prediction Model
### Semester Project | Machine Learning – Regression

**Dataset:** King County House Sales (`kc_house_data.csv`)

**Project goal:** Build and evaluate machine learning regression models that estimate a house's sale price from its available property features.

> **How to use this notebook:** Upload `kc_house_data.csv` to the Colab session when prompted, then run cells from top to bottom. Read the explanations and replace any project details your instructor asks you to personalize.

### Bilal Farooqi-70177313
### Sheraz Jaffer-70176504

## 1. Problem Definition

### 1.1 Problem statement
House prices depend on several factors, including living area, location, number of bedrooms and bathrooms, condition, grade, and construction year. Estimating a reasonable price manually can be difficult. This project uses historical house-sale data to train a supervised machine learning regression model that predicts the sale price of a house.

### 1.2 Target variable
- **Target (`y`):** `price`
- **Input features (`X`):** property characteristics such as bedrooms, bathrooms, living area, lot size, floors, waterfront, view, condition, grade, year built, renovation year, and location.

### 1.3 Objectives
1. Understand and clean the dataset.
2. Explore distributions, relationships, and unusual values.
3. Prepare useful features without leaking information from the test set.
4. Train multiple regression algorithms.
5. Compare models using MAE, RMSE, and R².
6. save the best complete preprocessing-and-model pipeline for later use.

### 1.4 Scope and assumptions
- The dataset contains historical sales from King County, Washington, USA.
- The model learns patterns from this dataset; it is not a professional property appraisal.
- Predictions may not generalize to other cities, countries, or market periods.
- The target is the recorded sale price, not a future guaranteed market value.

### 1.5 Success criteria
The project will compare models on an untouched test set. The selected model should have lower MAE/RMSE and a higher R² than the baseline model, while also being practical to save and use for new predictions. There is no guaranteed universal accuracy threshold; results will be reported honestly.

## 2. Data Collection and Dataset Description

The supplied CSV file is the project dataset. The next cell uploads it into Google Colab. If the file is already available in the Colab session, you can set `DATA_PATH` directly instead.

In [3]:
# Google Colab file upload
from google.colab import files
uploaded = files.upload()

# Select the uploaded CSV file
DATA_PATH = next((name for name in uploaded.keys() if name.lower().endswith(".csv")), None)
if DATA_PATH is None:
    raise FileNotFoundError("Please upload kc_house_data.csv")
print("Using dataset:", DATA_PATH)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape (rows, columns):", df.shape)
display(df.head())

### 2.1 Feature dictionary

| Feature | Meaning |
|---|---|
| `price` | Recorded house sale price; prediction target |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms |
| `sqft_living` | Interior living area in square feet |
| `sqft_lot` | Lot/land area in square feet |
| `floors` | Number of floors |
| `waterfront` | Whether the property has a waterfront view (0/1) |
| `view` | View rating |
| `condition` | Overall condition rating |
| `grade` | Construction/design grade |
| `sqft_above` | Area above ground |
| `sqft_basement` | Basement area |
| `yr_built` | Year the house was built |
| `yr_renovated` | Year of renovation; zero commonly means no recorded renovation |
| `zipcode` | Postal code; treated as a category, not a measured quantity |
| `lat`, `long` | Geographic latitude and longitude |
| `sqft_living15` | Living area of the nearest 15 neighbours (dataset field) |
| `sqft_lot15` | Lot area of the nearest 15 neighbours (dataset field) |
| `date` | Sale date |
| `id` | Record identifier; not a useful property characteristic for prediction |

## 3. Data Understanding

We inspect column names, data types, summary statistics, and target distribution before making preprocessing decisions.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nNumerical summary:")
display(df.describe().T)

print("\\nTarget variable summary:")
display(df["price"].describe())

In [ ]:
# Check target and feature distributions
plt.figure(figsize=(9, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of House Sale Prices")
plt.xlabel("Price")
plt.ylabel("Number of houses")
plt.tight_layout()
plt.show()

print("Skewness of price:", round(df["price"].skew(), 3))

## 4. Data Quality Checks

We check missing values, duplicate rows, invalid target values, and suspicious feature values. We do not automatically delete unusual records: an unusual value must be investigated and any decision documented.

In [ ]:
# Missing values by column
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

# Duplicate records
print("Duplicate rows:", df.duplicated().sum())

# Target checks
print("Missing target values:", df["price"].isna().sum())
print("Non-positive prices:", (df["price"] <= 0).sum())

# Investigate suspicious or uncommon values if these columns exist
for col in ["bedrooms", "bathrooms", "floors", "yr_built", "yr_renovated"]:
    if col in df.columns:
        print(f"\\n{col} value counts (top 15):")
        display(df[col].value_counts(dropna=False).head(15).to_frame("count"))

# Inspect records with unusually high bedroom counts
if "bedrooms" in df.columns:
    display(df.loc[df["bedrooms"] >= 10].sort_values("bedrooms", ascending=False).head(10))

### 4.1 Data-quality decisions

- Rows without a valid target price cannot be used for supervised training; they will be excluded from the modeling copy.
- Duplicate rows are reviewed. Exact duplicate records can be removed from the modeling copy.
- Missing numeric values will be imputed using the **training data median** inside a pipeline.
- Missing categorical values will be imputed using the **most frequent training value** inside a pipeline.
- Zero in `yr_renovated` is retained as a meaningful indication that no renovation year is recorded; a separate renovation indicator will be created.
- Unusual values such as a very high bedroom count will be investigated. We will not remove them solely because they look unusual.

In [ ]:
# Create a modeling copy and apply basic record-level cleaning
data = df.copy()

# Normalize column labels (remove accidental spaces)
data.columns = data.columns.str.strip()

# Remove exact duplicate rows
data = data.drop_duplicates()

# Remove records where the target is missing or non-positive
data = data[data["price"].notna() & (data["price"] > 0)].copy()

print("Original shape:", df.shape)
print("After basic cleaning:", data.shape)
print("Missing values after record-level cleaning:")
display(data.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_count"))

## 5. Initial Preprocessing and Feature Engineering

We convert the sale date into a datetime value and derive useful date features. We also create house age and renovation indicators. The original identifier and raw date are not passed directly to the model.

**Leakage prevention:** the train/test split is performed before fitting imputers, encoders, scalers, or models. These transformations are placed inside a scikit-learn pipeline so they learn only from training folds.

In [ ]:
# Convert sale date safely
if "date" in data.columns:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")

    # Extract sale year and month. Invalid dates remain missing and are imputed later if needed.
    data["sale_year"] = data["date"].dt.year
    data["sale_month"] = data["date"].dt.month

# Create age and renovation features using the sale year when available
if "yr_built" in data.columns:
    reference_year = data["sale_year"] if "sale_year" in data.columns else pd.Series(2015, index=data.index)
    data["house_age_at_sale"] = reference_year - data["yr_built"]

if "yr_renovated" in data.columns:
    data["has_been_renovated"] = (data["yr_renovated"].fillna(0) > 0).astype(int)

# Drop identifier and raw date; they are not used as direct model inputs
data = data.drop(columns=["id", "date"], errors="ignore")

print("Prepared dataset shape:", data.shape)
display(data.head())

## 6. Exploratory Data Analysis (EDA)

EDA helps us understand the data before modeling. We examine numerical distributions, relationships with price, correlations, and possible outliers. Visual patterns are associations, not proof of causation.

In [ ]:
# Numeric feature distributions
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
data[numeric_cols].hist(figsize=(16, 12), bins=30, edgecolor="black")
plt.suptitle("Distributions of Numerical Features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
corr = data.select_dtypes(include=np.number).corr(numeric_only=True)
target_corr = corr["price"].sort_values(ascending=False)
display(target_corr.to_frame("correlation_with_price"))

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Relationship between selected features and price
selected_features = [c for c in ["sqft_living", "grade", "bathrooms", "bedrooms", "house_age_at_sale"] if c in data.columns]

for feature in selected_features:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=data, x=feature, y="price", alpha=0.35)
    plt.title(f"{feature} vs House Price")
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare price across selected categorical/discrete features
for feature in [c for c in ["waterfront", "condition", "view"] if c in data.columns]:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=data, x=feature, y="price")
    plt.title(f"House Price by {feature}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots to help inspect extreme numerical values
for feature in [c for c in ["price", "sqft_living", "sqft_lot", "bedrooms"] if c in data.columns]:
    plt.figure(figsize=(8, 2.5))
    sns.boxplot(x=data[feature])
    plt.title(f"Outlier Inspection: {feature}")
    plt.tight_layout()
    plt.show()

### 6.1 EDA interpretation notes

After running the charts, write observations in your own words. For example:
- Is the price distribution symmetric or right-skewed?
- Which numerical features have the strongest positive relationship with price?
- Does living area generally increase with price?
- Are waterfront or higher-grade houses associated with different price ranges?
- Which columns contain extreme values, and are they plausible?

Do not claim that correlation proves causation.

## 7. Feature Preparation for Modeling

We separate the target from the input features. The postal code is handled as a categorical feature. Numeric and categorical columns receive different preprocessing steps.

In [ ]:
from sklearn.model_selection import train_test_split

# Ensure target is numeric and separate X and y
data["price"] = pd.to_numeric(data["price"], errors="coerce")
data = data.dropna(subset=["price"]).copy()

X = data.drop(columns=["price"])
y = data["price"]

# Treat zipcode as a category because postal codes are labels, not quantities
if "zipcode" in X.columns:
    X["zipcode"] = X["zipcode"].astype("string")

# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Number of input features:", X.shape[1])
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Split first: test data remains untouched until final evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

## 8. Reproducible Preprocessing Pipeline

The pipeline performs:
- Numeric missing-value imputation using the median.
- Numeric scaling for algorithms that benefit from scaling.
- Categorical missing-value imputation using the most frequent value.
- One-hot encoding for categorical variables.
- Model training after preprocessing.

Because preprocessing is inside the pipeline, cross-validation does not learn preprocessing statistics from validation folds.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 9. Baseline and Model Development

We first build a simple baseline that always predicts the training-set median price. Then we compare several regression algorithms.

**Colab performance note:** This notebook uses sequential processing (`n_jobs=1`) and a smaller cross-validation/search setup to reduce memory and CPU overload. Three-fold cross-validation is used to compare models and tune the selected algorithm. All preprocessing remains inside the pipeline and is fitted only on training folds.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Use fewer trees and sequential execution to avoid Colab RAM/CPU overload.
models = {
    "Baseline (Median)": DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42, min_samples_leaf=5
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=60, random_state=42, n_jobs=1, min_samples_leaf=2
    )
}

# 3-fold CV is a practical balance between reliability and execution time.
cv = KFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

cv_results = []
pipelines = {}

for name, model in models.items():
    print("Evaluating:", name)

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipelines[name] = pipe

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False
    )

    cv_results.append({
        "Model": name,
        "CV MAE": -scores["test_mae"].mean(),
        "CV RMSE": -scores["test_rmse"].mean(),
        "CV R2": scores["test_r2"].mean()
    })

cv_comparison = pd.DataFrame(cv_results).sort_values("CV RMSE")
display(cv_comparison)

### 9.1 Hyperparameter tuning

We tune the Random Forest using randomized search and cross-validation. This can take a few minutes in Colab. The search uses only the training data.

In [ ]:
# Hyperparameter tuning for Random Forest.
# Sequential execution and a small search reduce Colab resource usage.

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ))
])

param_distributions = {
    "model__n_estimators": [50, 80, 100],
    "model__max_depth": [15, 25, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", 1.0]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=4,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)
print("Best cross-validation RMSE:", -random_search.best_score_)

best_tuned_model = random_search.best_estimator_

## 10. Final Testing and Evaluation

The test set has not been used for fitting or hyperparameter tuning. We now evaluate the selected tuned model on this unseen data.

In [ ]:
# Fit the tuned pipeline on the complete training set
best_tuned_model.fit(X_train, y_train)

# Predict on unseen test data
y_pred = best_tuned_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

final_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Value": [mae, mse, rmse, r2]
})
display(final_metrics)

print(f"MAE:  ${mae:,.2f}")
print(f"MSE:  {mse:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

In [ ]:
# Actual vs predicted plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.35)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], "r--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")
plt.tight_layout()
plt.show()

# Residual analysis
residuals = y_test - y_pred
plt.figure(figsize=(7, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Distribution of Prediction Errors (Residuals)")
plt.xlabel("Actual - Predicted")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect a sample of prediction errors
error_analysis = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": y_pred,
    "Absolute Error": np.abs(y_test - y_pred)
})
display(error_analysis.sort_values("Absolute Error", ascending=False).head(10))

## 11. Model Comparison and Selection

Use the cross-validation table to compare candidate models, and use the final test metrics to report how the tuned model performs on unseen data.

- **MAE:** average absolute prediction error, in the same currency units as price.
- **MSE:** average squared error; gives larger errors more weight.
- **RMSE:** square root of MSE, in the same units as price.
- **R²:** proportion of target variation explained by the model; it can be negative if the model performs worse than a mean-based reference on the test set.

The final model should be selected using both predictive performance and practical considerations. Do not select a model based only on training performance.

## 12. Save the Final Model Pipeline

We save the complete pipeline, including imputers, encoders, scaler, and trained estimator. This is important because new data must receive exactly the same preprocessing as training data.

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)
MODEL_PATH = "models/house_price_prediction_pipeline.pkl"

joblib.dump(best_tuned_model, MODEL_PATH)
print("Saved model to:", MODEL_PATH)

# End-to-end prediction test using a few held-out rows
sample_input = X_test.head(3).copy()
sample_predictions = best_tuned_model.predict(sample_input)

display(pd.DataFrame({
    "Predicted Price": sample_predictions
}, index=sample_input.index))

# Download model artifact from Colab if needed
from google.colab import files
files.download(MODEL_PATH)

## 13. Deployment Preparation (Streamlit / Hugging Face)

The saved `.pkl` file is the model artifact needed by a small prediction application. The application should collect the same input features used by the notebook and call `pipeline.predict()`.

**Important deployment checks:**
1. Use the exact input column names and compatible data types.
2. Do not manually one-hot encode or scale app inputs; the saved pipeline already does that.
3. Validate user inputs (for example, non-negative area and reasonable bedroom counts).
4. Test at least one known sample from the test set.
5. Clearly label outputs as estimates, not guaranteed sale prices.

A separate `app.py` and `requirements.txt` can be created after the notebook is reviewed and accepted.

## 14. Monitoring and Retraining Considerations

After deployment:
- Track prediction errors when actual sale prices later become available.
- Monitor changes in house prices, locations, and feature distributions.
- Watch for new missing categories or input values.
- Retrain periodically with recent, reliable data.
- Keep a record of dataset version, model version, training date, and evaluation metrics.
- Do not automatically retrain on unverified or poor-quality data.

## 15. Limitations and Ethical Considerations

- The dataset represents a particular region and historical period; performance may be weaker in other locations or future markets.
- Some important influences (interest rates, economic conditions, property-specific details) may not be present.
- Location and historical market patterns can introduce bias.
- A prediction is an estimate and should not replace a qualified property valuation.
- Data source, cleaning choices, model settings, and limitations should be disclosed in the final report.

## 16. Final Conclusion

Complete this section after running all cells. Report:
1. Number of records and features used.
2. Main data-cleaning decisions.
3. Most important EDA findings.
4. Models compared and best hyperparameters.
5. Final test MAE, RMSE, and R².
6. Whether the tuned model improved on the baseline.
7. Main limitations and possible future improvements.

**Final result:** The project demonstrates an end-to-end supervised regression workflow, from raw data inspection and preprocessing to model evaluation and saving a reusable prediction pipeline.

## Appendix: Reproducibility

- Random seed: `42`
- Test size: `20%`
- Cross-validation: `5-fold` for model comparison and `3-fold` during randomized search
- Target: `price`
- Primary comparison metric: RMSE (with MAE and R² also reported)

If you change the random seed, split, features, or model settings, rerun the notebook from the beginning and document the change.